# Synthetic Data Augmentation for Medical Audio Classification
**Reproduction + Extension of McShannon et al. (2025)**

| File | Mô tả |
|------|-------|
| `01_data_preprocessing.py` | Load Coswara, mel-spectrogram, GroupShuffleSplit |
| `02_generative_model.py` | Train VAE / WGAN-GP / Diffusion |
| `03_evaluate_fad.py` | Fréchet Audio Distance — chất lượng dữ liệu sinh |
| `04_classifier.py` | Train CNN (4 scenarios) + lưu models |
| `05_ensemble.py` | Ensemble 4 models → kết quả cuối |

---

## 0. Setup

In [ ]:
# Mount Google Drive và chuyển vào thư mục dự án
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/MEDICAL-PROJECT')
print('Working directory:', os.getcwd())
print('Files:', os.listdir())

In [ ]:
# Kiểm tra GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cài đặt thư viện nếu cần
!pip install -q librosa scikit-learn seaborn

---
## 1. Data Preprocessing

- Load Coswara dataset (cough recordings)
- Filter: healthy (label=0) vs COVID+ (label=1)
- Mel-spectrogram: 128 mels, 0–8000 Hz, FFT=2048, hop=512
- **GroupShuffleSplit** theo user_id (tránh data leakage)
- Z-score normalization trên tập train
- Output: `dataset.pt`

In [ ]:
!python 01_data_preprocessing.py

In [ ]:
# Kiểm tra và visualize dataset
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

data = torch.load('dataset.pt', map_location='cpu', weights_only=False)

y_train = data['y_train'].numpy()
y_val   = data['y_val'].numpy()
y_test  = data['y_test'].numpy()

print('=== Dataset Summary ===')
for split, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    n_healthy = (y == 0).sum()
    n_covid   = (y == 1).sum()
    print(f'  {split:5s}: {len(y):4d} samples | '
          f'Healthy={n_healthy} ({n_healthy/len(y)*100:.1f}%) | '
          f'COVID+={n_covid} ({n_covid/len(y)*100:.1f}%)')

# Visualize class distribution
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('Class Distribution per Split', fontsize=13, fontweight='bold')
for ax, (split, y) in zip(axes, [('Train', y_train), ('Val', y_val), ('Test', y_test)]):
    counts = [( y==0).sum(), (y==1).sum()]
    bars = ax.bar(['Healthy', 'COVID+'], counts,
                  color=['#1565C0', '#E53935'], edgecolor='white', linewidth=1.2)
    for bar, c in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(c), ha='center', va='bottom', fontweight='bold')
    ax.set_title(split, fontsize=11)
    ax.set_ylabel('Count')
    ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('plot_00_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Visualize sample mel-spectrograms (Healthy vs COVID+)
X_train = data['X_train']  # (N, H, W, 1)

healthy_idx = (y_train == 0).nonzero()[0]
covid_idx   = (y_train == 1).nonzero()[0]

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
fig.suptitle('Sample Mel-Spectrograms: Healthy (top) vs COVID+ (bottom)',
             fontsize=13, fontweight='bold')

for i, ax in enumerate(axes[0]):
    spec = X_train[healthy_idx[i]].squeeze().numpy()
    im = ax.imshow(spec, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(f'Healthy #{i+1}', fontsize=10)
    ax.axis('off')

for i, ax in enumerate(axes[1]):
    spec = X_train[covid_idx[i]].squeeze().numpy()
    ax.imshow(spec, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(f'COVID+ #{i+1}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('plot_01_sample_spectrograms.png', dpi=120, bbox_inches='tight')
plt.show()
print('Spectrogram shape:', X_train[0].shape, '(H × W × 1)')

---
## 2. Train Generative Models

3 mô hình sinh được train **chỉ trên COVID+ samples**:
- **VAE**: 200 epochs, Adam lr=1e-4, β=0.1
- **WGAN-GP**: 300 epochs, RMSprop lr=5e-5, λ_gp=10, n_critic=5
- **Diffusion**: 400 epochs, AdamW lr=1e-4, T=1000 steps

Mỗi model sinh **558 synthetic samples** (50% of 1116 COVID+ train samples).

> ⚠️ Training lần đầu mất 2–4 giờ trên T4 GPU. Nếu resume từ checkpoint, dùng `--resume`.

In [ ]:
# Train VAE (200 epochs)
!python 02_generative_model.py --model vae

In [ ]:
# Train WGAN-GP (300 epochs)
!python 02_generative_model.py --model wgan

In [ ]:
# Train Diffusion (400 epochs) — lần đầu
!python 02_generative_model.py --model diffusion

In [ ]:
# Resume Diffusion từ checkpoint (nếu bị ngắt giữa chừng)
# Thay epoch number cho phù hợp
# !python 02_generative_model.py --model diffusion --resume diffusion_epoch_200.pth

In [ ]:
# Kiểm tra các file đã sinh
for fname in ['vae_samples.pt', 'gan_samples.pt', 'diffusion_samples.pt']:
    if os.path.exists(fname):
        t = torch.load(fname, map_location='cpu')
        print(f'  {fname}: {t.shape} | min={t.min():.3f} max={t.max():.3f}')
    else:
        print(f'  {fname}: NOT FOUND')

In [ ]:
# Visualize synthetic samples so sánh với real
real_covid = data['X_train'][y_train == 1]  # (N, H, W, 1)

sample_files = {
    'VAE':       'vae_samples.pt',
    'WGAN-GP':   'gan_samples.pt',
    'Diffusion': 'diffusion_samples.pt',
}

loaded = {}
for name, path in sample_files.items():
    if os.path.exists(path):
        loaded[name] = torch.load(path, map_location='cpu')

n_cols = 5
n_rows = 1 + len(loaded)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3 * n_rows))
fig.suptitle('Real vs Synthetic COVID+ Mel-Spectrograms', fontsize=13, fontweight='bold')

# Real samples
for j in range(n_cols):
    spec = real_covid[j].squeeze().numpy()
    axes[0, j].imshow(spec, aspect='auto', origin='lower', cmap='magma')
    axes[0, j].set_title(f'Real #{j+1}' if j > 0 else 'Real COVID+', fontsize=9)
    axes[0, j].axis('off')

# Synthetic samples
for i, (name, samples) in enumerate(loaded.items(), 1):
    if samples.ndim == 4 and samples.shape[1] == 1:  # (N,1,H,W)
        samples_show = samples
    else:
        samples_show = samples.permute(0,3,1,2) if samples.ndim==4 else samples.unsqueeze(1)
    for j in range(n_cols):
        spec = samples_show[j, 0].numpy()
        axes[i, j].imshow(spec, aspect='auto', origin='lower', cmap='magma')
        axes[i, j].set_title(f'{name} #{j+1}' if j > 0 else name, fontsize=9)
        axes[i, j].axis('off')

plt.tight_layout()
plt.savefig('plot_02_synthetic_spectrograms.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 3. Fréchet Audio Distance (FAD)

Đánh giá **chất lượng** dữ liệu sinh — FAD thấp hơn = gần real hơn.

Feature extractor: ResNet18 pretrained (ImageNet), conv1 adapted 3ch→1ch bằng cách average weights.

In [ ]:
!python 03_evaluate_fad.py

In [ ]:
# Visualize FAD scores
# !! Cập nhật các giá trị FAD thực tế sau khi chạy file 03 !!
fad_baseline = None   # Real vs Real split (lower bound)
fad_scores   = {
    'VAE':       None,  # thay bằng số thực tế
    'WGAN-GP':   None,
    'Diffusion': None,
}

# Đọc từ output của file 03 và điền vào đây, ví dụ:
# fad_baseline = 12.34
# fad_scores = {'VAE': 45.2, 'WGAN-GP': 38.7, 'Diffusion': 28.1}

if all(v is not None for v in fad_scores.values()) and fad_baseline is not None:
    fig, ax = plt.subplots(figsize=(8, 5))
    models   = list(fad_scores.keys())
    scores   = list(fad_scores.values())
    colors   = ['#1565C0', '#E53935', '#2E7D32']

    bars = ax.bar(models, scores, color=colors, edgecolor='white', linewidth=1.2, width=0.5)
    ax.axhline(fad_baseline, color='#F57F17', linestyle='--', linewidth=2.0,
               label=f'Real vs Real (lower bound) = {fad_baseline:.2f}')

    for bar, s in zip(bars, scores):
        gap = s - fad_baseline
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{s:.2f}\n(+{gap:.2f})', ha='center', va='bottom',
                fontsize=11, fontweight='bold')

    ax.set_ylabel('Fréchet Audio Distance (lower = better)', fontsize=12)
    ax.set_title('Generative Model Quality — FAD Score', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.savefig('plot_03_fad_scores.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('Chưa có FAD scores. Chạy file 03 và điền kết quả vào cell này.')

---
## 4. CNN Classifier Training

4 scenarios:
1. **Baseline** — chỉ real data
2. **Real + VAE** — thêm 558 VAE synthetic COVID+
3. **Real + WGAN-GP** — thêm 558 GAN synthetic COVID+
4. **Real + Diffusion** — thêm 558 Diffusion synthetic COVID+

CNN: 4 ConvBlocks (32→64→128→256), Adam lr=0.001, CosineAnnealing, early stopping patience=15.

In [ ]:
!python 04_classifier.py

In [ ]:
# Đọc và hiển thị kết quả file 04
import json

with open('results.json') as f:
    results_04 = json.load(f)

print('=== Classifier Results (from results.json) ===')
print(f'{"Scenario":<22} | {"macro F1":>8} | {"AUROC":>6}')
print('-' * 45)
for name, m in results_04.items():
    print(f'{name:<22} | {m["macro_F1"]:>8.4f} | {m["AUROC"]:>6.4f}')
print('\nPaper baseline: F1=0.645, AUROC=0.745')

---
## 5. Ensemble

Equation (1) từ bài báo:
$$P_{\text{ensemble}}(\text{COVID+}) = \frac{1}{4} \sum_{i=1}^{4} p_i(\text{COVID+})$$

**Kết quả bài báo gốc**: Ensemble F1=**0.664**, AUROC=**0.761**

In [ ]:
!python 05_ensemble.py

In [ ]:
# Hiển thị plot kết quả ensemble
from IPython.display import Image
Image('ensemble_results.png', width=900)

In [ ]:
# Đọc và in bảng so sánh cuối cùng
with open('ensemble_results.json') as f:
    ens_results = json.load(f)

paper = {
    'Baseline':       (0.645, 0.745),
    'Real+VAE':       (0.646, 0.748),
    'Real+WGAN-GP':   (0.609, 0.726),
    'Real+Diffusion': (0.644, 0.746),
    'Ensemble':       (0.664, 0.761),
}

print('=' * 70)
print(f'{"FINAL COMPARISON — Reproduced vs Paper":^70}')
print('=' * 70)
print(f'{"Scenario":<22} | {"Repro F1":>8} | {"Paper F1":>8} | {"Δ":>6} | {"Repro AUC":>9}')
print('-' * 70)
for name, m in ens_results.items():
    pf1, pauc = paper.get(name, (None, None))
    delta = f'{m["macro_F1"] - pf1:+.3f}' if pf1 else '  N/A'
    pf1s  = f'{pf1:.3f}' if pf1 else '  N/A'
    print(f'{name:<22} | {m["macro_F1"]:>8.4f} | {pf1s:>8} | {delta:>6} | {m["AUROC"]:>9.4f}')
print('=' * 70)

---
## 6. Analysis & Extension

Phần này **không có trong bài báo gốc** — phân tích sâu hơn.

In [ ]:
# t-SNE: visualize embedding của real vs synthetic
from sklearn.manifold import TSNE
import torch.nn as nn

class ConvBlock(nn.Sequential):
    def __init__(self, in_c, out_c):
        super().__init__(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

class AudioCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features   = nn.Sequential(ConvBlock(1,32), ConvBlock(32,64),
                                        ConvBlock(64,128), ConvBlock(128,256))
        self.pool       = nn.AdaptiveAvgPool2d((1,1))
        self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(256,2))
    def forward(self, x):
        return self.classifier(self.pool(self.features(x)).flatten(1))
    def embed(self, x):
        return self.pool(self.features(x)).flatten(1)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load baseline model
baseline_model = AudioCNN().to(DEVICE)
baseline_model.load_state_dict(torch.load('model_Baseline.pth', map_location=DEVICE))
baseline_model.eval()

def get_embeddings(model, tensors, label_val, max_n=200):
    tensors = tensors[:max_n]
    if tensors.ndim == 4 and tensors.shape[-1] == 1:
        tensors = tensors.permute(0,3,1,2)
    elif tensors.ndim == 4 and tensors.shape[1] != 1:
        tensors = tensors
    with torch.no_grad():
        emb = model.embed(tensors.to(DEVICE)).cpu().numpy()
    return emb, np.full(len(emb), label_val)

# Real COVID+ samples
real_covid_X = data['X_train'][y_train == 1]
emb_real, lbl_real = get_embeddings(baseline_model, real_covid_X, 0)

emb_list = [emb_real]
lbl_list = [lbl_real]
name_map = {1: 'VAE', 2: 'WGAN-GP', 3: 'Diffusion'}

for i, (name, path) in enumerate(['vae_samples.pt', 'gan_samples.pt', 'diffusion_samples.pt'], 1):
    if os.path.exists(path):
        s = torch.load(path, map_location='cpu')
        emb_s, lbl_s = get_embeddings(baseline_model, s, i)
        emb_list.append(emb_s)
        lbl_list.append(lbl_s)

if len(emb_list) > 1:
    all_emb = np.concatenate(emb_list)
    all_lbl = np.concatenate(lbl_list)

    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    emb_2d = tsne.fit_transform(all_emb)

    colors_tsne = ['#1565C0', '#E53935', '#FB8C00', '#2E7D32']
    labels_tsne = ['Real COVID+', 'VAE', 'WGAN-GP', 'Diffusion']

    fig, ax = plt.subplots(figsize=(9, 7))
    for lv in np.unique(all_lbl):
        mask = all_lbl == lv
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                   c=colors_tsne[int(lv)], label=labels_tsne[int(lv)],
                   alpha=0.6, s=25, edgecolors='none')
    ax.legend(fontsize=11, markerscale=2)
    ax.set_title('t-SNE: Real vs Synthetic COVID+ Embeddings\n(Baseline CNN feature space)',
                 fontsize=13, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('plot_04_tsne_embeddings.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('Cần ít nhất 1 file synthetic samples để vẽ t-SNE.')

In [ ]:
# Per-class F1 breakdown (Healthy vs COVID+)
# Đọc từ classification_report của file 05
# Cell này chạy lại ensemble nhanh để lấy per-class metrics

from sklearn.metrics import classification_report
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

class SimpleDS(Dataset):
    def __init__(self, X, y):
        self.X = X; self.y = y.long()
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = self.X[i]
        if x.shape[-1] == 1: x = x.permute(2,0,1)
        return x, self.y[i]

test_ds     = SimpleDS(data['X_test'], data['y_test'])
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)
true_labels = data['y_test'].numpy()

model_paths = {
    'Baseline':       'model_Baseline.pth',
    'Real+VAE':       'model_Real+VAE.pth',
    'Real+WGAN-GP':   'model_Real+WGAN-GP.pth',
    'Real+Diffusion': 'model_Real+Diffusion.pth',
}

prob_collection = []
fig, axes = plt.subplots(1, len(model_paths) + 1,
                          figsize=(5*(len(model_paths)+1), 4))
fig.suptitle('Per-class F1: Healthy vs COVID+', fontsize=13, fontweight='bold')

all_scenario_probs = []
for ax, (name, path) in zip(axes, model_paths.items()):
    if not os.path.exists(path):
        ax.set_title(f'{name}\n(missing)')
        continue
    m = AudioCNN().to(DEVICE)
    m.load_state_dict(torch.load(path, map_location=DEVICE))
    m.eval()
    all_probs, all_preds = [], []
    with torch.no_grad():
        for x, _ in test_loader:
            out = m(x.to(DEVICE))
            p   = torch.softmax(out, 1)[:, 1]
            all_probs.extend(p.cpu().numpy())
            all_preds.extend(out.argmax(1).cpu().numpy())
    all_scenario_probs.append(np.array(all_probs))
    from sklearn.metrics import f1_score as f1
    healthy_f1 = f1(true_labels, all_preds, average=None)[0]
    covid_f1   = f1(true_labels, all_preds, average=None)[1]
    ax.bar(['Healthy', 'COVID+'], [healthy_f1, covid_f1],
           color=['#1565C0', '#E53935'], edgecolor='white')
    ax.set_ylim(0, 1)
    ax.set_title(name, fontsize=10)
    ax.set_ylabel('F1 Score')
    for i, v in enumerate([healthy_f1, covid_f1]):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold', fontsize=10)
    ax.spines[['top','right']].set_visible(False)

# Ensemble per-class F1
if all_scenario_probs:
    ens_probs = np.mean(all_scenario_probs, axis=0)
    ens_preds = (ens_probs >= 0.5).astype(int)
    healthy_f1 = f1(true_labels, ens_preds, average=None)[0]
    covid_f1   = f1(true_labels, ens_preds, average=None)[1]
    ax = axes[-1]
    ax.bar(['Healthy', 'COVID+'], [healthy_f1, covid_f1],
           color=['#1565C0', '#E53935'], edgecolor='white')
    ax.set_ylim(0, 1)
    ax.set_title('Ensemble ★', fontsize=10, fontweight='bold')
    ax.set_ylabel('F1 Score')
    for i, v in enumerate([healthy_f1, covid_f1]):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold', fontsize=10)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('plot_05_perclass_f1.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Summary

Tổng hợp tất cả output files.

In [ ]:
# Liệt kê tất cả output files
import glob

print('=== Output Files ===')
categories = {
    'Data':        ['dataset.pt'],
    'Generative':  ['vae_samples.pt', 'gan_samples.pt', 'diffusion_samples.pt',
                    'vae_final.pth', 'wgan_gen_final.pth', 'diffusion_final.pth'],
    'Classifiers': glob.glob('model_*.pth'),
    'Results':     ['results.json', 'ensemble_results.json'],
    'Plots':       sorted(glob.glob('plot_*.png')) + ['ensemble_results.png'],
}

for cat, files in categories.items():
    print(f'\n  [{cat}]')
    for f in files:
        exists = os.path.exists(f)
        size   = f'{os.path.getsize(f)/1e6:.1f} MB' if exists else '—'
        status = '✓' if exists else '✗'
        print(f'    {status} {f:<40} {size}')

print('\n=== Final Results Summary ===')
if os.path.exists('ensemble_results.json'):
    with open('ensemble_results.json') as f:
        final = json.load(f)
    paper = {'Baseline':(0.645,0.745),'Real+VAE':(0.646,0.748),
             'Real+WGAN-GP':(0.609,0.726),'Real+Diffusion':(0.644,0.746),'Ensemble':(0.664,0.761)}
    print(f'{"Scenario":<22} | {"F1":>6} | {"Paper":>6} | {"AUROC":>6}')
    print('-' * 50)
    for name, m in final.items():
        pf1 = paper.get(name, (None,None))[0]
        pstr = f'{pf1:.3f}' if pf1 else ' N/A'
        print(f'{name:<22} | {m["macro_F1"]:>6.4f} | {pstr:>6} | {m["AUROC"]:>6.4f}')